In [ ]:
from google.colab import userdata
import os

# 1. Install MuJoCo & IK solver
!pip install -q mujoco mink "qpsolvers[osqp]"
!pip install -q "rerun-sdk[all]==0.28.2" "imageio[ffmpeg]"
!pip install -q "transformers<5.0.0" "lerobot==0.4.3"
!pip install -q hydra-core lightning omegaconf h5py hdf5plugin
!pip install -q stable-pretraining stable-worldmodel torchcodec
!pip install -q nnsight
!pip install -q fastapi uvicorn opencv-python

# 2. Clone Repo
!git clone --recursive https://github.com/anonymouscorl5-cmyk/le-probe.git

print("✅ Environment Initialized.")
exit()

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Harvest Activations

In [ ]:
!hf auth login

In [ ]:
%cd /content/le-probe/interpretability/transcoders
%env PYTHONPATH=/env/python:/content/le-probe/interpretability/transcoders:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
# !cp /content/drive/.../lewm_grasp_baseline/gr1_reward_tuned_v2.ckpt .
# !cp /content/drive/.../lewm_grasp_multiview/gr1_reward_tuned_v2.ckpt .
# !cp /content/drive/.../lewm_grasp_multiview_skeleton/gr1_reward_tuned_v2.ckpt .
!cp /content/drive/.../lewm_grasp_multiview_skeleton_dino/gr1_reward_tuned_v2.ckpt .

In [ ]:
!python /content/le-probe/dataset/skeleton/generate_priors.py gr1_pickup_grasp

In [ ]:
!python /content/le-probe/dataset/skeleton/verify_tiling.py gr1_pickup_grasp/videos/observation.images.world_center_tiled/chunk-000/file-000.mp4

In [ ]:
!python /content/le-probe/dataset/skeleton/generate_dino_priors.py

In [ ]:
!python /content/le-probe/dataset/skeleton/cache_fused_dataset.py

In [ ]:
!python /content/le-probe/dataset/skeleton/verify_cache.py

In [ ]:
# !python harvest_activations.py \
#     --output activations_granular \
#     --workers 4
# !python harvest_activations.py \
#     --model gr1_reward_tuned_v2.ckpt \
#     --output_dir activations_granular \
#     --workers 4 \
#     --multi_view
# !python harvest_activations.py \
#     --model gr1_reward_tuned_v6.ckpt \
#     --output_dir activations_granular \
#     --workers 2 \
#     --multi_view \
#     --use_skeleton
!python harvest_activations.py \
    --model gr1_reward_tuned_v1.ckpt \
    --output_dir activations_granular \
    --workers 2 \
    --multi_view \
    --use_skeleton \
    --use_dino


In [ ]:
# !python audit_harvest.py --dir activations_granular
# !python audit_harvest.py --dir activations_granular --multi_view
# !python audit_harvest.py --model gr1_reward_tuned_v6.ckpt --dir activations_granular --multi_view --use_skeleton
!python audit_harvest.py --model gr1_reward_tuned_v1.ckpt --dir activations_granular --multi_view --use_skeleton --use_dino

In [ ]:
# !cp -r activations_granular /content/drive/...lewm_grasp_baseline/run2/activations_granular
# !cp -r activations_granular /content/drive/...lewm_grasp_multiview/run2/activations_granular
# !cp -r activations_granular /content/drive/...lewm_grasp_multiview_skeleton/run2/activations_granular
!cp -r activations_granular /content/drive/...lewm_grasp_multiview_skeleton_dino/run4/activations_granular

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

### Train Transcoder

In [ ]:
%cd /content/le-probe/interpretability/transcoders
%env PYTHONPATH=/env/python:/content/le-probe/interpretability/transcoders:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
# !cp -r /content/drive/.../lewm_grasp_baseline/run2/activations_granular .
# !cp -r /content/drive/.../lewm_grasp_baseline/run2/gr1_reward_tuned_v2.ckpt .
# !cp -r /content/drive/.../lewm_grasp_multiview/run2/activations_granular .
# !cp -r /content/drive/.../lewm_grasp_multiview/run2/gr1_reward_tuned_v2.ckpt .
# !cp -r /content/drive/.../lewm_grasp_multiview_skeleton/run2/activations_granular .
# !cp -r /content/drive/.../lewm_grasp_multiview_skeleton/run2/gr1_reward_tuned_v6.ckpt .
!cp -r /content/drive/.../lewm_grasp_multiview_skeleton_dino/run4/activations_granular .
!cp -r /content/drive/.../lewm_grasp_multiview_skeleton_dino/run4/gr1_reward_tuned_v1.ckpt .

In [ ]:
!EPOCHS=5 ENCODER_BATCH_SIZE=16384 bash batch_train.sh

In [ ]:
# !cp -r transcoder_weights_residual /content/drive/.../lewm_grasp_baseline/run1/transcoder_weights_residual
# !cp -r transcoder_weights_residual /content/drive/.../lewm_grasp_multiview/run2/transcoder_weights_residual
# !cp -r transcoder_weights_residual /content/drive/.../lewm_grasp_multiview_skeleton/run2/transcoder_weights_residual
!cp -r transcoder_weights_residual/* /content/drive/.../lewm_grasp_multiview_skeleton_dino/run4/transcoder_weights_residual

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

### Neuronpedia Integration

In [ ]:
!hf auth login

In [ ]:
%cd /content/le-probe/interpretability/dashboard
%env PYTHONPATH=/env/python:/content/le-probe:/content/le-probe/interpretability/transcoders:/content/le-probe/interpretability/dashboard:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
!cp -r /content/drive/.../transcoder_weights_residual .
!cp -r /content/drive/.../activations_granular .
!cp -r /content/drive/.../gr1_reward_tuned_v2.ckpt .

In [ ]:
!python engine.py \
    --repo gr1_pickup_grasp \
    --meta activations_granular/encoder_L0.json \
    --model gr1_reward_tuned_v2.ckpt \
    --transcoders transcoder_weights_residual \
    --min-k 10

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

### Visualize Manifold

In [ ]:
%cd /content/le-probe/interpretability/manifold
%env PYTHONPATH=/env/python:/content/le-probe:/content/le-probe/interpretability/transcoders:/content/le-probe/interpretability/manifold:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
# !cp -r /content/drive/.../lewm_grasp_baseline/run2/gr1_reward_tuned_v2.ckpt .
# !cp -r /content/drive/.../lewm_grasp_multiview/run2/gr1_reward_tuned_v2.ckpt .
# !cp -r /content/drive/.../lewm_grasp_multiview_skeleton/run2/gr1_reward_tuned_v6.ckpt .
!cp -r /content/drive/.../lewm_grasp_multiview_skeleton_dino/run4/gr1_reward_tuned_v1.ckpt .

In [ ]:
!python le-probe/dataset/skeleton/generate_priors.py gr1_pickup_grasp

In [ ]:
!python le-probe/dataset/skeleton/verify_tiling.py gr1_pickup_grasp/videos/observation.images.world_center_tiled/chunk-000/file-000.mp4

In [ ]:
# !python harvest_manifold.py --model gr1_reward_tuned_v2.ckpt --episodes 200
# !python harvest_manifold.py --model gr1_reward_tuned_v2.ckpt --episodes 200 --multi_view
# !python harvest_manifold.py \
#     --model gr1_reward_tuned_v6.ckpt \
#     --episodes 200 \
#     --multi_view \
#     --use_skeleton
!python le-probe/interpretability/manifold/harvest_manifold.py \
    --model gr1_reward_tuned_v1.ckpt \
    --episodes 200 \
    --use_skeleton \
    --multi_view \
    --use_dino

In [ ]:
# !cp manifold_data.pt /content/drive/.../lewm_grasp_baseline/run2/manifold_data_v2.pt
# !cp manifold_data.pt /content/drive/.../lewm_grasp_multiview/run2/manifold_data.pt
# !cp manifold_data.pt /content/drive/.../lewm_grasp_multiview_skeleton/run2/manifold_data.pt
# !cp manifold_data.pt /content/drive/.../lewm_grasp_multiview_skeleton_dino/run3/manifold_data.pt
!cp manifold_data.pt /content/drive/.../lewm_grasp_multiview_skeleton_dino/run4/manifold_data.pt

In [ ]:
from google.colab import drive
drive.flush_and_unmount()